# RoadCrack AI · GPU training
Select a GPU runtime. Use your authorized datasets and stable persistent storage. Run D00/D10/D20 detection and binary segmentation as separate tasks. Use the commit containing these files. No metrics are included.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
WORK = Path("/content/drive/MyDrive/RoadCrackAI")
REPO = Path("/content/RoadCrack-AI")
# Clone your accessible repository into REPO and place prepared datasets in WORK.
assert (REPO / "requirements.txt").is_file(), "Check out the project into /content/RoadCrack-AI"
%cd /content/RoadCrack-AI


In [ ]:
import os, subprocess, sys
from pathlib import Path
os.environ["PYTHONPATH"] = str(REPO / "src")
# Keep Colab's matched CUDA Torch/torchvision. Install other pinned packages.
req = Path("/content/roadcrack-colab-requirements.txt")
req.write_text("\n".join(line for line in Path("requirements.txt").read_text().splitlines()
                         if not line.startswith(("torch==", "torchvision==", "--extra-index"))))
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req)], check=True)
subprocess.run([sys.executable, "scripts/check_env.py"], check=True)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
def run(module, *args):
    subprocess.run([sys.executable, "-m", f"roadcrack.{module}", *map(str, args)], check=True)


Build datasets in this runtime from reviewed manifests or update the absolute dataset.yaml path after restoring a copy. Run fits one at a time. For detection, resume with the original run directory and fit/weights/last.pt. For segmentation, append --resume with last.pt and reuse the same hyperparameters.

In [ ]:
det = WORK / "processed-detection"
seg = WORK / "processed-segmentation"
runs = WORK / "runs"
weights = WORK / "initial-weights"
# Stage authorized official weights at these paths before running.
# Change the variant in a new run to compare yolo11s and yolo26m.
run("detection", "train", "--weights", weights / "yolo26s.pt", "--data", det,
    "--run", runs / "fit/det-01", "--device", "0", "--size", 640)


In [ ]:
run("segmentation", "train", "--architecture", "UnetPlusPlus", "--pretrained",
    "--data", seg, "--run", runs / "fit/seg-01", "--device", "cuda:0")
# Repeat with DeepLabV3Plus in a new run for the architecture comparison.


In [ ]:
# Run only after the configuration and threshold are fixed using validation.
run("detection", "evaluate", "--weights", runs / "fit/det-01/fit/weights/best.pt",
    "--data", det, "--run", runs / "det-01", "--split", "test", "--device", "0")
run("segmentation", "evaluate", "--resume", runs / "fit/seg-01/best.pt",
    "--data", seg, "--run", runs / "seg-01", "--device", "cuda:0")
subprocess.run([sys.executable, "scripts/compare_runs.py", "--runs", str(runs),
                "--output", str(WORK / "evaluation.json")], check=True)
